# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_06 — VaR, CVaR, and Stress Testing

### Research question

How do we estimate portfolio downside risk using VaR, CVaR / Expected Shortfall, backtest those estimates, and design stress scenarios that reveal risks not captured by ordinary volatility?

This notebook follows:

```text
04_01_multi_asset_return_engine
04_03_volatility_targeting_and_position_sizing
04_04_mean_variance_optimization_ledoit_wolf
04_05_pca_spectral_risk_analysis
```

The previous notebook decomposed portfolio risk into PCA modes. This notebook asks:

> What could the portfolio lose, how often does it exceed its risk forecast, and what happens under extreme historical or hypothetical scenarios?

It covers:

1. Value at Risk;
2. Conditional Value at Risk / Expected Shortfall;
3. historical VaR;
4. Gaussian parametric VaR;
5. Cornish-Fisher modified VaR;
6. EWMA volatility VaR;
7. Monte Carlo VaR;
8. rolling VaR forecasts;
9. VaR exceedance backtesting;
10. Kupiec-style unconditional coverage test;
11. Expected Shortfall diagnostics;
12. component VaR / marginal risk approximation;
13. historical stress scenarios;
14. hypothetical factor shocks;
15. PCA stress scenarios;
16. stress loss attribution;
17. drawdown and crisis diagnostics;
18. governance checklist;
19. saved outputs and manifest.

The key idea:

> VaR estimates a quantile. CVaR estimates tail severity. Stress testing asks what happens when the model is wrong.

## 1. VaR definition

Let portfolio return be $R_t$, and loss be:

$$
L_t=-R_t
$$

At confidence level $\alpha$, Value at Risk is:

$$
VaR_\alpha = q_\alpha(L)
$$

where $q_\alpha$ is the $\alpha$-quantile of loss.

A 95% one-day VaR of 2% means:

> Under the model or historical sample, losses should exceed 2% on about 5% of days.

Important:

> VaR says little about how bad losses are after the VaR threshold is breached.

## 2. CVaR / Expected Shortfall definition

Conditional Value at Risk, also called Expected Shortfall, is:

$$
CVaR_\alpha = E[L \mid L \ge VaR_\alpha]
$$

It measures average loss in the tail beyond the VaR threshold.

CVaR is often more informative than VaR because it tells us about tail severity.

For a portfolio:

$$
L_t = -w^\top r_t
$$

historical CVaR is estimated by averaging the worst losses beyond the VaR quantile.

## 3. Stress testing

VaR and CVaR are estimated from a distribution or historical sample.

Stress testing asks:

> What if the future is worse or different from the past?

Stress tests can be:

1. **Historical**  
   Replay crisis-like days.

2. **Hypothetical**  
   Shock equity, rates, commodity, FX, and crypto factors.

3. **PCA-based**  
   Shock along dominant covariance eigenmodes.

4. **Reverse stress**  
   Find the shock required to lose a chosen amount.

5. **Liquidity-adjusted**  
   Widen costs and reduce capacity during stress.

This notebook implements several simple but useful versions.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import NormalDist
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class TailRiskConfig:
    n_days: int = 2_400
    annualisation: int = 252
    confidence_levels: tuple = (0.95, 0.99)
    rolling_window: int = 252
    ewma_lambda: float = 0.94
    n_mc_scenarios: int = 50_000
    stress_sigma: float = 3.0
    initial_capital: float = 1_000_000.0
    seed: int = 42


config = TailRiskConfig()
config

## 5. Synthetic multi-asset universe

We simulate a universe with:

- equities;
- bonds;
- commodities;
- FX carry;
- trend-following;
- real assets;
- crypto proxy.

The simulation includes:

1. correlated factors;
2. volatility regimes;
3. stress jumps;
4. fat tails;
5. changing correlations.

This creates a realistic playground for tail-risk methods.

In [ ]:
def simulate_tail_risk_universe(config: TailRiskConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(config.seed)

    dates = pd.bdate_range("2014-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND_FOLLOW",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND_FOLLOW": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_ann = np.array([0.14, 0.055, 0.13, 0.08, 0.09, 0.35])
    factor_vol_daily = factor_vol_ann / np.sqrt(config.annualisation)

    factor_corr = np.array([
        [ 1.00, -0.25,  0.25,  0.15, -0.10,  0.35],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20],
        [ 0.25,  0.10,  1.00,  0.10,  0.05,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15],
        [-0.10,  0.15,  0.05,  0.00,  1.00,  0.00],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00],
    ])

    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.75]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND_FOLLOW", "trend"] = 1.00
    loadings.loc["TREND_FOLLOW", "market"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.35
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.30

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.08,
        "EM_EQ": 0.13,
        "US_BOND": 0.025,
        "EU_BOND": 0.030,
        "GOLD": 0.12,
        "OIL": 0.22,
        "COPPER": 0.18,
        "FX_CARRY": 0.07,
        "TREND_FOLLOW": 0.08,
        "REIT": 0.10,
        "BTC_PROXY": 0.50,
    })

    annual_mean = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.06,
        "EM_EQ": 0.08,
        "US_BOND": 0.025,
        "EU_BOND": 0.020,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "FX_CARRY": 0.030,
        "TREND_FOLLOW": 0.050,
        "REIT": 0.060,
        "BTC_PROXY": 0.120,
    })

    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    factor_returns = np.zeros((config.n_days, len(factor_names)))
    asset_returns = np.zeros((config.n_days, len(assets)))
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.3
        cov_t = factor_cov * vol_multiplier**2

        # Fat-tailed factor shock via scaled t.
        z = rng.standard_t(df=6, size=len(factor_names)) * np.sqrt((6 - 2) / 6)
        L = np.linalg.cholesky(cov_t + 1e-12 * np.eye(len(factor_names)))
        f = L @ z

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
        elif u < 0.013:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.040)
            f[2] += rng.uniform(0.030, 0.090)
            f[0] -= rng.uniform(0.010, 0.050)
        elif u < 0.018:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)
        elif u < 0.023:
            stress_type[t] = "crypto_crash"
            f[5] -= rng.uniform(0.120, 0.300)
            f[0] -= rng.uniform(0.005, 0.035)

        eps = rng.standard_t(df=5, size=len(assets)) * np.sqrt((5 - 2) / 5)
        eps = eps * (idio_vol_ann.values / np.sqrt(config.annualisation))

        mu_daily = annual_mean.values / config.annualisation
        asset_returns[t] = mu_daily + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    returns_df = pd.DataFrame(asset_returns, columns=assets)
    returns_df.insert(0, "date", dates)
    returns_df["regime"] = regime
    returns_df["stress_type"] = stress_type

    factors_df = pd.DataFrame(factor_returns, columns=factor_names)
    factors_df.insert(0, "date", dates)
    factors_df["regime"] = regime
    factors_df["stress_type"] = stress_type

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
        "annual_mean_assumption": [annual_mean[a] for a in assets],
    })

    return returns_df, factors_df, metadata


returns_df, factor_returns, asset_metadata = simulate_tail_risk_universe(config)
asset_cols = asset_metadata["asset"].tolist()
returns = returns_df.set_index("date")[asset_cols]

returns_df.head(), factor_returns.head(), asset_metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(returns.index, (1 + returns[asset]).cumprod(), label=asset, alpha=0.75)
plt.title("Synthetic Asset Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(returns_df["date"], returns_df["regime"], drawstyle="steps-post")
plt.title("Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Portfolio construction for risk testing

We define three example portfolios:

1. `balanced_multi_asset`;
2. `risk_on`;
3. `defensive`.

These are static weights so we can focus on tail-risk measurement.

In [ ]:
def build_example_portfolios(asset_cols):
    weights = pd.DataFrame(0.0, index=asset_cols, columns=["balanced_multi_asset", "risk_on", "defensive"])

    weights.loc[:, "balanced_multi_asset"] = {
        "US_EQ": 0.16,
        "EU_EQ": 0.12,
        "EM_EQ": 0.08,
        "US_BOND": 0.16,
        "EU_BOND": 0.10,
        "GOLD": 0.10,
        "OIL": 0.06,
        "COPPER": 0.05,
        "FX_CARRY": 0.05,
        "TREND_FOLLOW": 0.06,
        "REIT": 0.04,
        "BTC_PROXY": 0.02,
    }

    weights.loc[:, "risk_on"] = {
        "US_EQ": 0.22,
        "EU_EQ": 0.16,
        "EM_EQ": 0.14,
        "US_BOND": 0.04,
        "EU_BOND": 0.02,
        "GOLD": 0.05,
        "OIL": 0.08,
        "COPPER": 0.07,
        "FX_CARRY": 0.05,
        "TREND_FOLLOW": 0.04,
        "REIT": 0.08,
        "BTC_PROXY": 0.05,
    }

    weights.loc[:, "defensive"] = {
        "US_EQ": 0.08,
        "EU_EQ": 0.06,
        "EM_EQ": 0.03,
        "US_BOND": 0.30,
        "EU_BOND": 0.20,
        "GOLD": 0.14,
        "OIL": 0.02,
        "COPPER": 0.02,
        "FX_CARRY": 0.03,
        "TREND_FOLLOW": 0.10,
        "REIT": 0.02,
        "BTC_PROXY": 0.00,
    }

    # Ensure columns sum to one.
    weights = weights.div(weights.sum(axis=0), axis=1)

    return weights


portfolio_weights = build_example_portfolios(asset_cols)
portfolio_weights

In [ ]:
portfolio_returns = pd.DataFrame(index=returns.index)

for portfolio in portfolio_weights.columns:
    portfolio_returns[portfolio] = returns @ portfolio_weights[portfolio]

portfolio_losses = -portfolio_returns

portfolio_returns.head()

## 7. Basic portfolio diagnostics

Before tail-risk estimation, inspect ordinary risk metrics:

- annualised return;
- annualised volatility;
- skew;
- excess kurtosis;
- max drawdown;
- worst day.

In [ ]:
def max_drawdown_from_returns(r):
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1
    return dd, float(dd.min())


def basic_portfolio_diagnostics(portfolio_returns):
    rows = []

    for col in portfolio_returns.columns:
        r = portfolio_returns[col]
        dd, mdd = max_drawdown_from_returns(r)

        rows.append({
            "portfolio": col,
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "skew": float(r.skew()),
            "excess_kurtosis": float(r.kurtosis()),
            "max_drawdown": mdd,
            "worst_day_return": float(r.min()),
            "best_day_return": float(r.max()),
        })

    return pd.DataFrame(rows).sort_values("annualised_vol", ascending=False)


basic_diag = basic_portfolio_diagnostics(portfolio_returns)
basic_diag

In [ ]:
plt.figure(figsize=(12, 6))
for col in portfolio_returns.columns:
    plt.plot(portfolio_returns.index, (1 + portfolio_returns[col]).cumprod(), label=col)
plt.title("Portfolio Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
for col in portfolio_returns.columns:
    plt.hist(portfolio_returns[col], bins=100, alpha=0.45, density=True, label=col)
plt.axvline(0, linestyle="--")
plt.title("Daily Portfolio Return Distributions")
plt.xlabel("Daily return")
plt.ylabel("Density")
plt.legend()
plt.show()

## 8. Historical VaR and CVaR

Historical VaR uses the empirical loss distribution.

For confidence level $\alpha$:

$$
VaR_\alpha = quantile_\alpha(L)
$$

$$
CVaR_\alpha = mean(L \mid L \ge VaR_\alpha)
$$

In [ ]:
def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


hist_rows = []

for portfolio in portfolio_losses.columns:
    losses = portfolio_losses[portfolio].dropna().to_numpy()

    for alpha in config.confidence_levels:
        var, cvar = historical_var_cvar(losses, alpha)
        hist_rows.append({
            "portfolio": portfolio,
            "method": "historical",
            "alpha": alpha,
            "VaR": var,
            "CVaR": cvar
        })

hist_var_cvar = pd.DataFrame(hist_rows)
hist_var_cvar

## 9. Gaussian parametric VaR and CVaR

Assume loss is normally distributed:

$$
L \sim N(\mu_L,\sigma_L^2)
$$

Then:

$$
VaR_\alpha = \mu_L + z_\alpha\sigma_L
$$

Expected Shortfall for normal loss:

$$
\begin{aligned}
CVaR_\alpha &= \mu_L \\
&\quad + \sigma_L \frac{\phi(z_\alpha)}{1-\alpha}
\end{aligned}
$$

where:

- $z_\alpha = \Phi^{-1}(\alpha)$;
- $\phi$ is the standard normal density.

In [ ]:
normal_dist = NormalDist()

def normal_pdf(x):
    return np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)


def gaussian_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    mu = losses.mean()
    sigma = losses.std(ddof=1)

    z = normal_dist.inv_cdf(alpha)
    var = mu + z * sigma
    cvar = mu + sigma * normal_pdf(z) / (1 - alpha)

    return float(var), float(cvar)


gauss_rows = []

for portfolio in portfolio_losses.columns:
    losses = portfolio_losses[portfolio].dropna().to_numpy()

    for alpha in config.confidence_levels:
        var, cvar = gaussian_var_cvar(losses, alpha)
        gauss_rows.append({
            "portfolio": portfolio,
            "method": "gaussian",
            "alpha": alpha,
            "VaR": var,
            "CVaR": cvar
        })

gaussian_var_cvar_table = pd.DataFrame(gauss_rows)
gaussian_var_cvar_table

## 10. Cornish-Fisher modified VaR

Gaussian VaR ignores skew and kurtosis.

Cornish-Fisher adjusts the normal quantile using sample skewness $S$ and excess kurtosis $K$:

$$
\begin{aligned}
z_{CF} &= z \\
&\quad + \frac{1}{6}(z^2-1)S \\
&\quad + \frac{1}{24}(z^3-3z)K \\
&\quad - \frac{1}{36}(2z^3-5z)S^2
\end{aligned}
$$

Then:

$$
VaR_{CF}=\mu_L+z_{CF}\sigma_L
$$

This is an approximation and can behave badly for extreme skew/kurtosis.

In [ ]:
def cornish_fisher_var(losses, alpha):
    losses = pd.Series(losses).dropna()
    mu = losses.mean()
    sigma = losses.std(ddof=1)
    skew = losses.skew()
    kurt = losses.kurtosis()

    z = normal_dist.inv_cdf(alpha)
    z_cf = (
        z
        + (1 / 6) * (z**2 - 1) * skew
        + (1 / 24) * (z**3 - 3 * z) * kurt
        - (1 / 36) * (2 * z**3 - 5 * z) * skew**2
    )

    var = mu + z_cf * sigma
    return float(var), float(z_cf), float(skew), float(kurt)


cf_rows = []

for portfolio in portfolio_losses.columns:
    losses = portfolio_losses[portfolio].dropna()

    for alpha in config.confidence_levels:
        var, z_cf, skew, kurt = cornish_fisher_var(losses, alpha)
        cf_rows.append({
            "portfolio": portfolio,
            "method": "cornish_fisher",
            "alpha": alpha,
            "VaR": var,
            "z_cf": z_cf,
            "loss_skew": skew,
            "loss_excess_kurtosis": kurt
        })

cornish_fisher_var_table = pd.DataFrame(cf_rows)
cornish_fisher_var_table

## 11. Compare VaR methods

We compare:

1. historical VaR;
2. Gaussian VaR;
3. Cornish-Fisher VaR.

CVaR is reported for historical and Gaussian methods.

In [ ]:
var_comparison = pd.concat([
    hist_var_cvar,
    gaussian_var_cvar_table,
    cornish_fisher_var_table.assign(CVaR=np.nan)[["portfolio", "method", "alpha", "VaR", "CVaR"]]
], ignore_index=True)

var_comparison.sort_values(["portfolio", "alpha", "VaR"])

In [ ]:
for alpha in config.confidence_levels:
    sub = var_comparison[var_comparison["alpha"] == alpha]

    plt.figure(figsize=(10, 6))
    for method, g in sub.groupby("method"):
        plt.plot(g["portfolio"], g["VaR"], marker="o", label=method)
    plt.title(f"VaR Comparison at {int(alpha * 100)}%")
    plt.xlabel("Portfolio")
    plt.ylabel("One-day loss VaR")
    plt.legend()
    plt.show()

## 12. Rolling VaR forecasts

A VaR estimate is only useful if it is available before the return occurs.

For each day $t$:

1. estimate VaR using returns through $t-1$;
2. compare to realised loss at $t$.

We implement rolling historical and Gaussian VaR.

In [ ]:
def rolling_var_forecasts(portfolio_returns, window, confidence_levels):
    rows = []

    for portfolio in portfolio_returns.columns:
        r = portfolio_returns[portfolio]
        losses = -r

        for t in range(window, len(r)):
            hist_window = losses.iloc[t - window:t]
            realised_loss = losses.iloc[t]
            date = losses.index[t]

            for alpha in confidence_levels:
                hist_var, hist_cvar = historical_var_cvar(hist_window, alpha)
                gauss_var, gauss_cvar = gaussian_var_cvar(hist_window, alpha)

                rows.append({
                    "date": date,
                    "portfolio": portfolio,
                    "alpha": alpha,
                    "realised_loss": realised_loss,
                    "historical_VaR": hist_var,
                    "historical_CVaR": hist_cvar,
                    "gaussian_VaR": gauss_var,
                    "gaussian_CVaR": gauss_cvar,
                    "historical_exceedance": realised_loss > hist_var,
                    "gaussian_exceedance": realised_loss > gauss_var
                })

    return pd.DataFrame(rows)


rolling_var = rolling_var_forecasts(portfolio_returns, config.rolling_window, config.confidence_levels)
rolling_var.head()

In [ ]:
example_portfolio = "balanced_multi_asset"
example_alpha = 0.95

plot_df = rolling_var[
    (rolling_var["portfolio"] == example_portfolio)
    & (rolling_var["alpha"] == example_alpha)
].copy()

plt.figure(figsize=(12, 6))
plt.plot(plot_df["date"], plot_df["realised_loss"], label="realised loss", alpha=0.65)
plt.plot(plot_df["date"], plot_df["historical_VaR"], label="historical VaR")
plt.plot(plot_df["date"], plot_df["gaussian_VaR"], label="gaussian VaR")
plt.title(f"Rolling VaR Forecasts: {example_portfolio}, {int(example_alpha*100)}%")
plt.xlabel("Date")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 13. VaR exceedance backtesting

For confidence level $\alpha$, expected exceedance rate is:

$$
1-\alpha
$$

If 95% VaR is calibrated, about 5% of days should exceed VaR.

Too many exceedances means VaR underestimates risk.

Too few may mean VaR is overly conservative or sample is too short.

In [ ]:
def var_backtest_summary(rolling_var):
    rows = []

    for (portfolio, alpha), g in rolling_var.groupby(["portfolio", "alpha"]):
        n = len(g)
        expected_rate = 1 - alpha

        for method in ["historical", "gaussian"]:
            exceed_col = f"{method}_exceedance"
            x = int(g[exceed_col].sum())
            observed_rate = x / n

            rows.append({
                "portfolio": portfolio,
                "alpha": alpha,
                "method": method,
                "n_obs": n,
                "n_exceedances": x,
                "expected_exceedance_rate": expected_rate,
                "observed_exceedance_rate": observed_rate,
                "expected_exceedances": n * expected_rate,
                "exceedance_ratio_observed_over_expected": observed_rate / expected_rate if expected_rate > 0 else np.nan
            })

    return pd.DataFrame(rows)


var_backtest = var_backtest_summary(rolling_var)
var_backtest

In [ ]:
plt.figure(figsize=(12, 6))
for method, g in var_backtest[var_backtest["alpha"] == 0.95].groupby("method"):
    plt.plot(g["portfolio"], g["observed_exceedance_rate"], marker="o", label=method)
plt.axhline(0.05, linestyle="--", label="expected 5%")
plt.title("Observed VaR Exceedance Rate, 95% VaR")
plt.xlabel("Portfolio")
plt.ylabel("Exceedance rate")
plt.legend()
plt.show()

## 14. Kupiec-style unconditional coverage test

Kupiec's likelihood ratio test checks whether observed exceedance rate equals expected rate.

If:

$$
x = \text{number of exceedances}
$$

$$
n = \text{number of forecasts}
$$

$$
p = 1-\alpha
$$

then under the null:

$$
x \sim Binomial(n,p)
$$

The likelihood ratio statistic is approximately:

$$
LR_{uc} = -2\log \frac{(1-p)^{n-x}p^x} {(1-\hat p)^{n-x}\hat p^x}
$$

where:

$$
\hat p = x/n
$$

This notebook reports the statistic and a rough chi-square p-value approximation.

In [ ]:
def chi_square_1_sf_approx(x):
    # Survival function for chi-square with 1 degree:
    # P(ChiSq_1 > x) = 2 * (1 - Phi(sqrt(x))).
    if x < 0 or not np.isfinite(x):
        return np.nan
    return 2 * (1 - normal_dist.cdf(np.sqrt(x)))


def kupiec_test(n, x, p):
    if n <= 0:
        return np.nan, np.nan

    phat = x / n

    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    phat = np.clip(phat, eps, 1 - eps)

    log_l_null = (n - x) * np.log(1 - p) + x * np.log(p)
    log_l_alt = (n - x) * np.log(1 - phat) + x * np.log(phat)

    lr = -2 * (log_l_null - log_l_alt)
    p_value = chi_square_1_sf_approx(lr)

    return float(lr), float(p_value)


kupiec_rows = []

for _, row in var_backtest.iterrows():
    lr, p_value = kupiec_test(
        n=int(row["n_obs"]),
        x=int(row["n_exceedances"]),
        p=1 - row["alpha"]
    )

    kupiec_rows.append({
        **row.to_dict(),
        "kupiec_lr_uc": lr,
        "kupiec_p_value_approx": p_value,
        "reject_5pct_approx": bool(p_value < 0.05) if pd.notna(p_value) else False
    })

kupiec_report = pd.DataFrame(kupiec_rows)
kupiec_report

## 15. Expected Shortfall diagnostics

For days exceeding VaR, compare realised tail losses to CVaR.

If average exceedance loss is consistently above CVaR, the model underestimates tail severity.

In [ ]:
def es_diagnostic(rolling_var):
    rows = []

    for (portfolio, alpha), g in rolling_var.groupby(["portfolio", "alpha"]):
        for method in ["historical", "gaussian"]:
            var_col = f"{method}_VaR"
            cvar_col = f"{method}_CVaR"
            exceed = g["realised_loss"] > g[var_col]
            exceed_losses = g.loc[exceed, "realised_loss"]

            rows.append({
                "portfolio": portfolio,
                "alpha": alpha,
                "method": method,
                "n_exceedances": int(exceed.sum()),
                "mean_realised_exceedance_loss": float(exceed_losses.mean()) if len(exceed_losses) else np.nan,
                "mean_forecast_CVaR_on_exceedance_days": float(g.loc[exceed, cvar_col].mean()) if exceed.sum() else np.nan,
                "tail_loss_minus_cvar": float(exceed_losses.mean() - g.loc[exceed, cvar_col].mean()) if exceed.sum() else np.nan
            })

    return pd.DataFrame(rows)


es_report = es_diagnostic(rolling_var)
es_report

## 16. EWMA Gaussian VaR

Rolling windows weight old and recent observations equally.

EWMA volatility gives more weight to recent returns:

$$
\begin{aligned}
\sigma_t^2 &= \lambda\sigma_{t-1}^2 \\
&\quad + (1-\lambda)r_{t-1}^2
\end{aligned}
$$

EWMA Gaussian VaR:

$$
VaR_{\alpha,t} = -\mu_t + z_\alpha\sigma_t
$$

where loss is negative return.

In [ ]:
def ewma_portfolio_vol(portfolio_returns, lam):
    vol = pd.DataFrame(index=portfolio_returns.index, columns=portfolio_returns.columns, dtype=float)

    for col in portfolio_returns.columns:
        r = portfolio_returns[col].fillna(0.0)
        var = np.zeros(len(r))
        var[0] = r.iloc[:30].var() if len(r) >= 30 else r.var()

        for t in range(1, len(r)):
            var[t] = lam * var[t - 1] + (1 - lam) * r.iloc[t - 1] ** 2

        vol[col] = np.sqrt(var)

    return vol


ewma_vol = ewma_portfolio_vol(portfolio_returns, config.ewma_lambda)

ewma_var_rows = []

for portfolio in portfolio_returns.columns:
    r = portfolio_returns[portfolio]
    mu_rolling = r.rolling(config.rolling_window, min_periods=60).mean().shift(1).fillna(0.0)
    sigma = ewma_vol[portfolio]

    for alpha in config.confidence_levels:
        z = normal_dist.inv_cdf(alpha)
        var_series = -mu_rolling + z * sigma

        df = pd.DataFrame({
            "date": r.index,
            "portfolio": portfolio,
            "alpha": alpha,
            "realised_loss": -r,
            "ewma_gaussian_VaR": var_series,
            "ewma_gaussian_exceedance": (-r) > var_series
        })

        ewma_var_rows.append(df)

ewma_var = pd.concat(ewma_var_rows, ignore_index=True)
ewma_var.head()

In [ ]:
ewma_backtest = (
    ewma_var
    .groupby(["portfolio", "alpha"])
    .agg(
        n_obs=("realised_loss", "count"),
        n_exceedances=("ewma_gaussian_exceedance", "sum"),
        observed_exceedance_rate=("ewma_gaussian_exceedance", "mean")
    )
    .reset_index()
)

ewma_backtest["expected_exceedance_rate"] = 1 - ewma_backtest["alpha"]
ewma_backtest

## 17. Monte Carlo VaR and CVaR

Monte Carlo VaR simulates losses from an estimated return distribution.

Here:

1. estimate mean and covariance from historical data;
2. draw multivariate normal scenarios;
3. compute portfolio losses;
4. estimate VaR and CVaR.

This is parametric Monte Carlo.

It is only as good as the assumed distribution.

In [ ]:
def monte_carlo_var_cvar(returns, weights, alpha, n_scenarios, seed):
    rng = np.random.default_rng(seed)

    mu = returns.mean().to_numpy()
    cov = returns.cov().to_numpy()

    scenarios = rng.multivariate_normal(mu, cov, size=n_scenarios)
    port_returns = scenarios @ weights
    losses = -port_returns

    var, cvar = historical_var_cvar(losses, alpha)

    return var, cvar, losses


mc_rows = []
mc_loss_samples = {}

for portfolio in portfolio_weights.columns:
    w = portfolio_weights[portfolio].to_numpy()

    for alpha in config.confidence_levels:
        var, cvar, losses = monte_carlo_var_cvar(
            returns,
            w,
            alpha,
            config.n_mc_scenarios,
            config.seed + int(alpha * 1000)
        )

        mc_rows.append({
            "portfolio": portfolio,
            "method": "monte_carlo_gaussian",
            "alpha": alpha,
            "VaR": var,
            "CVaR": cvar
        })

        mc_loss_samples[(portfolio, alpha)] = losses

mc_var_cvar = pd.DataFrame(mc_rows)
mc_var_cvar

In [ ]:
portfolio = "balanced_multi_asset"
alpha = 0.99

plt.figure(figsize=(10, 6))
plt.hist(mc_loss_samples[(portfolio, alpha)], bins=100, density=True)
plt.axvline(mc_var_cvar[(mc_var_cvar["portfolio"] == portfolio) & (mc_var_cvar["alpha"] == alpha)]["VaR"].iloc[0], linestyle="--", label="MC VaR")
plt.title(f"Monte Carlo Loss Distribution: {portfolio}, {int(alpha*100)}%")
plt.xlabel("Loss")
plt.ylabel("Density")
plt.legend()
plt.show()

## 18. Component VaR approximation

For portfolio weights $w$, portfolio volatility is:

$$
\sigma_p=\sqrt{w^\top\Sigma w}
$$

For Gaussian VaR:

$$
VaR_\alpha = z_\alpha\sigma_p - \mu_p
$$

Marginal contribution to volatility:

$$
MC_i = \frac{(\Sigma w)_i}{\sigma_p}
$$

Component volatility contribution:

$$
C_i = w_i MC_i
$$

Component VaR approximation:

$$
CVaR_i^{component} \approx \frac{C_i}{\sigma_p}VaR_\alpha
$$

This identifies which assets drive parametric VaR.

In [ ]:
def component_gaussian_var(returns, weights, alpha):
    mu = returns.mean().to_numpy()
    cov = returns.cov().to_numpy()
    w = np.asarray(weights)

    port_mu = w @ mu
    port_vol = np.sqrt(w.T @ cov @ w)
    z = normal_dist.inv_cdf(alpha)

    var = -port_mu + z * port_vol

    marginal_vol = cov @ w / max(port_vol, 1e-12)
    component_vol = w * marginal_vol
    component_var = component_vol / max(port_vol, 1e-12) * var

    out = pd.DataFrame({
        "asset": returns.columns,
        "weight": w,
        "marginal_vol": marginal_vol,
        "component_vol": component_vol,
        "component_var": component_var,
        "component_var_pct": component_var / var if var != 0 else np.nan
    })

    return var, out.sort_values("component_var", ascending=False)


component_var_tables = []
component_var_summary = []

for portfolio in portfolio_weights.columns:
    for alpha in config.confidence_levels:
        var, table = component_gaussian_var(returns, portfolio_weights[portfolio], alpha)
        table["portfolio"] = portfolio
        table["alpha"] = alpha
        component_var_tables.append(table)

        component_var_summary.append({
            "portfolio": portfolio,
            "alpha": alpha,
            "gaussian_VaR": var,
            "component_sum": table["component_var"].sum()
        })

component_var_table = pd.concat(component_var_tables, ignore_index=True)
component_var_summary = pd.DataFrame(component_var_summary)

component_var_summary

In [ ]:
plot_component = component_var_table[
    (component_var_table["portfolio"] == "risk_on")
    & (component_var_table["alpha"] == 0.99)
].sort_values("component_var")

plt.figure(figsize=(10, 6))
plt.barh(plot_component["asset"], plot_component["component_var"])
plt.title("Component Gaussian VaR: Risk-On Portfolio, 99%")
plt.xlabel("Component VaR")
plt.ylabel("Asset")
plt.show()

## 19. Historical stress scenarios

Historical stress scenarios are the worst realised days by portfolio loss.

For each portfolio, identify the largest losses and their asset-level contributions:

$$
contribution_{i,t}=w_i r_{i,t}
$$

In [ ]:
def historical_stress_days(returns, portfolio_weights, n_worst=10):
    rows = []
    contribution_rows = []

    for portfolio in portfolio_weights.columns:
        w = portfolio_weights[portfolio]
        port_returns = returns @ w
        losses = -port_returns
        worst_dates = losses.sort_values(ascending=False).head(n_worst).index

        for date in worst_dates:
            rows.append({
                "portfolio": portfolio,
                "date": date,
                "portfolio_return": port_returns.loc[date],
                "loss": losses.loc[date],
                "stress_type": returns_df.set_index("date").loc[date, "stress_type"],
                "regime": int(returns_df.set_index("date").loc[date, "regime"])
            })

            contrib = w * returns.loc[date]
            for asset, value in contrib.items():
                contribution_rows.append({
                    "portfolio": portfolio,
                    "date": date,
                    "asset": asset,
                    "asset_return": returns.loc[date, asset],
                    "weight": w[asset],
                    "return_contribution": value,
                    "loss_contribution": -value
                })

    return pd.DataFrame(rows), pd.DataFrame(contribution_rows)


historical_stress, historical_stress_contrib = historical_stress_days(returns, portfolio_weights, n_worst=10)

historical_stress.head()

In [ ]:
portfolio = "risk_on"
worst_date = historical_stress[historical_stress["portfolio"] == portfolio].iloc[0]["date"]

sub = historical_stress_contrib[
    (historical_stress_contrib["portfolio"] == portfolio)
    & (historical_stress_contrib["date"] == worst_date)
].sort_values("loss_contribution")

plt.figure(figsize=(10, 6))
plt.barh(sub["asset"], sub["loss_contribution"])
plt.axvline(0, linestyle="--")
plt.title(f"Loss Contribution on Worst Day: {portfolio}, {worst_date.date()}")
plt.xlabel("Loss contribution")
plt.ylabel("Asset")
plt.show()

## 20. Hypothetical factor stress scenarios

We define stylised scenarios:

1. equity crash;
2. inflation shock;
3. commodity crash;
4. crypto crash;
5. rates shock;
6. risk-on melt-up.

Each scenario shocks asset returns directly using intuitive mappings.

In [ ]:
def build_hypothetical_scenarios(asset_cols):
    scenarios = {
        "equity_crash": {
            "US_EQ": -0.08, "EU_EQ": -0.09, "EM_EQ": -0.12,
            "US_BOND": 0.015, "EU_BOND": 0.012,
            "GOLD": 0.010, "OIL": -0.040, "COPPER": -0.050,
            "FX_CARRY": -0.025, "TREND_FOLLOW": 0.020,
            "REIT": -0.080, "BTC_PROXY": -0.180,
        },
        "inflation_shock": {
            "US_EQ": -0.035, "EU_EQ": -0.040, "EM_EQ": -0.050,
            "US_BOND": -0.030, "EU_BOND": -0.028,
            "GOLD": 0.035, "OIL": 0.090, "COPPER": 0.070,
            "FX_CARRY": 0.010, "TREND_FOLLOW": 0.015,
            "REIT": -0.045, "BTC_PROXY": -0.060,
        },
        "commodity_crash": {
            "US_EQ": -0.015, "EU_EQ": -0.020, "EM_EQ": -0.030,
            "US_BOND": 0.006, "EU_BOND": 0.005,
            "GOLD": -0.025, "OIL": -0.120, "COPPER": -0.100,
            "FX_CARRY": -0.015, "TREND_FOLLOW": 0.020,
            "REIT": -0.015, "BTC_PROXY": -0.060,
        },
        "crypto_crash": {
            "US_EQ": -0.010, "EU_EQ": -0.012, "EM_EQ": -0.015,
            "US_BOND": 0.002, "EU_BOND": 0.002,
            "GOLD": 0.003, "OIL": -0.005, "COPPER": -0.005,
            "FX_CARRY": -0.005, "TREND_FOLLOW": 0.005,
            "REIT": -0.010, "BTC_PROXY": -0.300,
        },
        "rates_shock": {
            "US_EQ": -0.030, "EU_EQ": -0.032, "EM_EQ": -0.040,
            "US_BOND": -0.040, "EU_BOND": -0.038,
            "GOLD": -0.015, "OIL": -0.010, "COPPER": -0.015,
            "FX_CARRY": 0.005, "TREND_FOLLOW": 0.000,
            "REIT": -0.070, "BTC_PROXY": -0.080,
        },
        "risk_on_meltup": {
            "US_EQ": 0.050, "EU_EQ": 0.045, "EM_EQ": 0.060,
            "US_BOND": -0.010, "EU_BOND": -0.008,
            "GOLD": -0.005, "OIL": 0.030, "COPPER": 0.035,
            "FX_CARRY": 0.020, "TREND_FOLLOW": -0.010,
            "REIT": 0.040, "BTC_PROXY": 0.120,
        },
    }

    scenario_df = pd.DataFrame(scenarios).T.reindex(columns=asset_cols)
    return scenario_df


hypothetical_scenarios = build_hypothetical_scenarios(asset_cols)
hypothetical_scenarios

In [ ]:
def scenario_portfolio_losses(scenarios, portfolio_weights):
    rows = []
    contrib_rows = []

    for scenario_name, shock in scenarios.iterrows():
        for portfolio in portfolio_weights.columns:
            w = portfolio_weights[portfolio]
            port_return = float(w @ shock)
            loss = -port_return

            rows.append({
                "scenario": scenario_name,
                "portfolio": portfolio,
                "scenario_return": port_return,
                "scenario_loss": loss
            })

            for asset in asset_cols:
                contribution = w[asset] * shock[asset]
                contrib_rows.append({
                    "scenario": scenario_name,
                    "portfolio": portfolio,
                    "asset": asset,
                    "shock_return": shock[asset],
                    "weight": w[asset],
                    "return_contribution": contribution,
                    "loss_contribution": -contribution
                })

    return pd.DataFrame(rows), pd.DataFrame(contrib_rows)


scenario_losses, scenario_contrib = scenario_portfolio_losses(hypothetical_scenarios, portfolio_weights)
scenario_losses.sort_values("scenario_loss", ascending=False).head(12)

In [ ]:
plt.figure(figsize=(12, 6))
for portfolio in portfolio_weights.columns:
    sub = scenario_losses[scenario_losses["portfolio"] == portfolio]
    plt.plot(sub["scenario"], sub["scenario_loss"], marker="o", label=portfolio)
plt.xticks(rotation=45, ha="right")
plt.title("Hypothetical Scenario Losses")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 21. PCA stress scenarios

Using the asset correlation matrix, we create PCA shocks:

$$
shock^{(k)} = -a\sqrt{\lambda_k}q_k\sigma
$$

where:

- $q_k$ is the $k$-th eigenvector;
- $\lambda_k$ is the eigenvalue;
- $\sigma$ is asset volatility;
- $a$ is stress severity.

This shocks the portfolio along statistically dominant risk modes.

In [ ]:
def eigen_decomposition_symmetric(matrix, labels):
    eigvals, eigvecs = np.linalg.eigh(matrix)
    order = np.argsort(eigvals)[::-1]

    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    for j in range(eigvecs.shape[1]):
        max_idx = np.argmax(np.abs(eigvecs[:, j]))
        if eigvecs[max_idx, j] < 0:
            eigvecs[:, j] *= -1

    eig_table = pd.DataFrame({
        "component": np.arange(1, len(eigvals) + 1),
        "eigenvalue": eigvals,
        "explained_variance_ratio": eigvals / eigvals.sum(),
        "cumulative_explained_variance": np.cumsum(eigvals / eigvals.sum())
    })

    loadings = pd.DataFrame(
        eigvecs,
        index=labels,
        columns=[f"PC{i}" for i in range(1, len(eigvals) + 1)]
    )

    return eig_table, loadings


corr = returns.corr()
eig_corr, load_corr = eigen_decomposition_symmetric(corr.to_numpy(), asset_cols)

eig_corr.head()

In [ ]:
def pca_stress_scenarios(returns, eig_corr, load_corr, n_components=5, stress_sigma=3.0):
    asset_vol = returns.std(ddof=1)
    rows = []

    for k in range(1, n_components + 1):
        pc = f"PC{k}"
        eig = eig_corr.loc[eig_corr["component"] == k, "eigenvalue"].iloc[0]
        q = load_corr[pc]

        for direction in [-1, 1]:
            shock = direction * stress_sigma * np.sqrt(eig) * q * asset_vol

            for asset, value in shock.items():
                rows.append({
                    "scenario": f"{pc}_{'positive' if direction > 0 else 'negative'}_{stress_sigma:.1f}sigma",
                    "component": pc,
                    "direction": direction,
                    "asset": asset,
                    "shock_return": value
                })

    return pd.DataFrame(rows)


pca_scenarios_long = pca_stress_scenarios(returns, eig_corr, load_corr, n_components=5, stress_sigma=config.stress_sigma)
pca_scenarios = pca_scenarios_long.pivot(index="scenario", columns="asset", values="shock_return").reindex(columns=asset_cols)

pca_scenarios.head()

In [ ]:
pca_scenario_losses, pca_scenario_contrib = scenario_portfolio_losses(pca_scenarios, portfolio_weights)

pca_scenario_losses.sort_values("scenario_loss", ascending=False).head(10)

## 22. Reverse stress test

A reverse stress test asks:

> What size shock along a risk direction would create a specified portfolio loss?

For a shock vector $v$, portfolio return is:

$$
R_p(a)=a w^\top v
$$

To produce loss $L^*$:

$$
a = \frac{-L^*}{w^\top v}
$$

This is useful for identifying how much movement in a risk mode would breach a loss threshold.

In [ ]:
def reverse_stress_along_scenario(scenario_vector, portfolio_weights, target_loss):
    rows = []

    for portfolio in portfolio_weights.columns:
        w = portfolio_weights[portfolio]
        unit_return = float(w @ scenario_vector)

        if abs(unit_return) < 1e-12:
            multiplier = np.nan
        else:
            multiplier = -target_loss / unit_return

        rows.append({
            "portfolio": portfolio,
            "unit_scenario_return": unit_return,
            "target_loss": target_loss,
            "required_multiplier": multiplier
        })

    return pd.DataFrame(rows)


# Use negative first PCA scenario as unit.
unit_scenario = pca_scenarios.loc[pca_scenarios.index.str.contains("PC1_negative")].iloc[0]
reverse_stress = reverse_stress_along_scenario(unit_scenario, portfolio_weights, target_loss=0.10)

reverse_stress

## 23. Drawdown diagnostics

VaR is one-day risk.

Drawdown is path risk:

$$
DD_t = \frac{NAV_t}{\max_{\tau\le t}NAV_\tau}-1
$$

A risk report should include both.

In [ ]:
drawdown_rows = []
drawdown_series = pd.DataFrame(index=portfolio_returns.index)

for portfolio in portfolio_returns.columns:
    nav = (1 + portfolio_returns[portfolio]).cumprod()
    dd = nav / nav.cummax() - 1
    drawdown_series[portfolio] = dd

    worst_date = dd.idxmin()

    drawdown_rows.append({
        "portfolio": portfolio,
        "max_drawdown": float(dd.min()),
        "max_drawdown_date": str(worst_date),
        "return_on_max_drawdown_date": float(portfolio_returns.loc[worst_date, portfolio])
    })

drawdown_report = pd.DataFrame(drawdown_rows)
drawdown_report

In [ ]:
plt.figure(figsize=(12, 6))
for portfolio in drawdown_series.columns:
    plt.plot(drawdown_series.index, drawdown_series[portfolio], label=portfolio)
plt.title("Portfolio Drawdowns")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.show()

## 24. Tail risk dashboard

We combine:

- historical VaR/CVaR;
- Gaussian VaR/CVaR;
- Cornish-Fisher VaR;
- Monte Carlo VaR/CVaR;
- exceedance rates;
- drawdown;
- worst stress scenario.

In [ ]:
dashboard_rows = []

for portfolio in portfolio_weights.columns:
    hist_99 = hist_var_cvar[
        (hist_var_cvar["portfolio"] == portfolio)
        & (hist_var_cvar["alpha"] == 0.99)
    ].iloc[0]

    gauss_99 = gaussian_var_cvar_table[
        (gaussian_var_cvar_table["portfolio"] == portfolio)
        & (gaussian_var_cvar_table["alpha"] == 0.99)
    ].iloc[0]

    mc_99 = mc_var_cvar[
        (mc_var_cvar["portfolio"] == portfolio)
        & (mc_var_cvar["alpha"] == 0.99)
    ].iloc[0]

    backtest_95_hist = var_backtest[
        (var_backtest["portfolio"] == portfolio)
        & (var_backtest["alpha"] == 0.95)
        & (var_backtest["method"] == "historical")
    ].iloc[0]

    worst_hypo = scenario_losses[scenario_losses["portfolio"] == portfolio]["scenario_loss"].max()
    worst_pca = pca_scenario_losses[pca_scenario_losses["portfolio"] == portfolio]["scenario_loss"].max()

    dd = drawdown_report[drawdown_report["portfolio"] == portfolio].iloc[0]

    dashboard_rows.append({
        "portfolio": portfolio,
        "hist_VaR_99": hist_99["VaR"],
        "hist_CVaR_99": hist_99["CVaR"],
        "gaussian_VaR_99": gauss_99["VaR"],
        "gaussian_CVaR_99": gauss_99["CVaR"],
        "mc_VaR_99": mc_99["VaR"],
        "mc_CVaR_99": mc_99["CVaR"],
        "hist_95_exceedance_rate": backtest_95_hist["observed_exceedance_rate"],
        "max_drawdown": dd["max_drawdown"],
        "worst_hypothetical_scenario_loss": worst_hypo,
        "worst_pca_scenario_loss": worst_pca,
    })

tail_risk_dashboard = pd.DataFrame(dashboard_rows).sort_values("hist_CVaR_99", ascending=False)
tail_risk_dashboard

## 25. Risk governance flags

We create simple flags:

1. VaR exceedance ratio too high;
2. CVaR much larger than VaR;
3. stress loss exceeds threshold;
4. drawdown exceeds threshold.

These are examples of dashboard governance rules.

In [ ]:
governance_rows = []

for _, row in tail_risk_dashboard.iterrows():
    portfolio = row["portfolio"]

    hist_95 = var_backtest[
        (var_backtest["portfolio"] == portfolio)
        & (var_backtest["alpha"] == 0.95)
        & (var_backtest["method"] == "historical")
    ].iloc[0]

    exceed_ratio = hist_95["exceedance_ratio_observed_over_expected"]
    tail_ratio = row["hist_CVaR_99"] / row["hist_VaR_99"] if row["hist_VaR_99"] != 0 else np.nan
    worst_stress = max(row["worst_hypothetical_scenario_loss"], row["worst_pca_scenario_loss"])

    governance_rows.append({
        "portfolio": portfolio,
        "exceedance_ratio_95_hist": exceed_ratio,
        "tail_ratio_CVaR99_to_VaR99": tail_ratio,
        "worst_stress_loss": worst_stress,
        "max_drawdown": row["max_drawdown"],
        "flag_var_underestimation": bool(exceed_ratio > 1.5),
        "flag_fat_tail": bool(tail_ratio > 1.5),
        "flag_stress_loss_gt_10pct": bool(worst_stress > 0.10),
        "flag_drawdown_gt_20pct": bool(row["max_drawdown"] < -0.20),
    })

governance_flags = pd.DataFrame(governance_rows)
governance_flags

## 26. Practical checklist for VaR, CVaR, and stress testing

Before trusting a tail-risk report, check:

1. **Loss convention**  
   Is loss positive when return is negative?

2. **Confidence level**  
   Are 95% and 99% clearly distinguished?

3. **Method**  
   Historical, Gaussian, modified, Monte Carlo, or model-based?

4. **Backtesting**  
   Do exceedance rates match expectations?

5. **Tail severity**  
   Is CVaR much larger than VaR?

6. **Stress scenarios**  
   Are scenarios economically meaningful?

7. **PCA scenarios**  
   Do statistical shocks reveal hidden factor concentration?

8. **Drawdown**  
   Does one-day VaR miss path-dependent risk?

9. **Regime dependence**  
   Does VaR underestimate risk in volatile regimes?

10. **Governance action**  
   What happens if a flag is breached?

## 27. Saving outputs

The notebook saves:

1. synthetic returns;
2. factor returns;
3. asset metadata;
4. portfolio weights;
5. portfolio returns and losses;
6. basic diagnostics;
7. historical VaR/CVaR;
8. Gaussian VaR/CVaR;
9. Cornish-Fisher VaR;
10. rolling VaR forecasts;
11. VaR backtest reports;
12. Kupiec test report;
13. ES diagnostics;
14. EWMA VaR;
15. Monte Carlo VaR/CVaR;
16. component VaR;
17. stress scenarios;
18. PCA stress scenarios;
19. drawdown report;
20. tail-risk dashboard;
21. governance flags;
22. manifest.

In [ ]:
output_dir = Path("data/processed/var_cvar_and_stress_testing_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
factor_returns_path = output_dir / "factor_returns.csv"
asset_metadata_path = output_dir / "asset_metadata.csv"
portfolio_weights_path = output_dir / "portfolio_weights.csv"
portfolio_returns_path = output_dir / "portfolio_returns.csv"
portfolio_losses_path = output_dir / "portfolio_losses.csv"
basic_diag_path = output_dir / "basic_portfolio_diagnostics.csv"
hist_var_cvar_path = output_dir / "historical_var_cvar.csv"
gaussian_var_cvar_path = output_dir / "gaussian_var_cvar.csv"
cornish_fisher_path = output_dir / "cornish_fisher_var.csv"
var_comparison_path = output_dir / "var_comparison.csv"
rolling_var_path = output_dir / "rolling_var_forecasts.csv"
var_backtest_path = output_dir / "var_backtest.csv"
kupiec_report_path = output_dir / "kupiec_report.csv"
es_report_path = output_dir / "expected_shortfall_diagnostics.csv"
ewma_var_path = output_dir / "ewma_gaussian_var.csv"
ewma_backtest_path = output_dir / "ewma_backtest.csv"
mc_var_cvar_path = output_dir / "monte_carlo_var_cvar.csv"
component_var_path = output_dir / "component_var.csv"
component_var_summary_path = output_dir / "component_var_summary.csv"
historical_stress_path = output_dir / "historical_stress_days.csv"
historical_stress_contrib_path = output_dir / "historical_stress_contributions.csv"
hypothetical_scenarios_path = output_dir / "hypothetical_scenarios.csv"
scenario_losses_path = output_dir / "scenario_losses.csv"
scenario_contrib_path = output_dir / "scenario_contributions.csv"
pca_scenarios_path = output_dir / "pca_scenarios.csv"
pca_scenario_losses_path = output_dir / "pca_scenario_losses.csv"
pca_scenario_contrib_path = output_dir / "pca_scenario_contributions.csv"
reverse_stress_path = output_dir / "reverse_stress.csv"
drawdown_report_path = output_dir / "drawdown_report.csv"
drawdown_series_path = output_dir / "drawdown_series.csv"
dashboard_path = output_dir / "tail_risk_dashboard.csv"
governance_flags_path = output_dir / "governance_flags.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
factor_returns.to_csv(factor_returns_path, index=False)
asset_metadata.to_csv(asset_metadata_path, index=False)
portfolio_weights.to_csv(portfolio_weights_path)
portfolio_returns.to_csv(portfolio_returns_path)
portfolio_losses.to_csv(portfolio_losses_path)
basic_diag.to_csv(basic_diag_path, index=False)
hist_var_cvar.to_csv(hist_var_cvar_path, index=False)
gaussian_var_cvar_table.to_csv(gaussian_var_cvar_path, index=False)
cornish_fisher_var_table.to_csv(cornish_fisher_path, index=False)
var_comparison.to_csv(var_comparison_path, index=False)
rolling_var.to_csv(rolling_var_path, index=False)
var_backtest.to_csv(var_backtest_path, index=False)
kupiec_report.to_csv(kupiec_report_path, index=False)
es_report.to_csv(es_report_path, index=False)
ewma_var.to_csv(ewma_var_path, index=False)
ewma_backtest.to_csv(ewma_backtest_path, index=False)
mc_var_cvar.to_csv(mc_var_cvar_path, index=False)
component_var_table.to_csv(component_var_path, index=False)
component_var_summary.to_csv(component_var_summary_path, index=False)
historical_stress.to_csv(historical_stress_path, index=False)
historical_stress_contrib.to_csv(historical_stress_contrib_path, index=False)
hypothetical_scenarios.to_csv(hypothetical_scenarios_path)
scenario_losses.to_csv(scenario_losses_path, index=False)
scenario_contrib.to_csv(scenario_contrib_path, index=False)
pca_scenarios.to_csv(pca_scenarios_path)
pca_scenario_losses.to_csv(pca_scenario_losses_path, index=False)
pca_scenario_contrib.to_csv(pca_scenario_contrib_path, index=False)
reverse_stress.to_csv(reverse_stress_path, index=False)
drawdown_report.to_csv(drawdown_report_path, index=False)
drawdown_series.to_csv(drawdown_series_path)
tail_risk_dashboard.to_csv(dashboard_path, index=False)
governance_flags.to_csv(governance_flags_path, index=False)

manifest = {
    "dataset_name": "var_cvar_and_stress_testing_outputs",
    "schema_version": "var_cvar_and_stress_testing_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "portfolios": portfolio_weights.columns.tolist(),
    "confidence_levels": list(config.confidence_levels),
    "tail_risk_dashboard": tail_risk_dashboard.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "notes": [
        "Loss is defined as negative return, so positive values represent losses.",
        "Historical, Gaussian, Cornish-Fisher, EWMA Gaussian, and Monte Carlo VaR are included.",
        "CVaR / Expected Shortfall is included for historical, Gaussian, and Monte Carlo methods.",
        "Rolling VaR forecasts are shifted by construction because each forecast uses only the previous window.",
        "Kupiec unconditional coverage test is implemented with a chi-square(1) approximation.",
        "Stress testing includes historical worst days, hypothetical scenarios, PCA mode shocks, and reverse stress.",
        "This notebook is educational and synthetic; production VaR requires governance, data quality controls, and model validation."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, var_comparison_path, dashboard_path, governance_flags_path, manifest_path

## 28. Limitations

### 28.1 Synthetic data

The notebook uses synthetic returns and stress events.

Real portfolios need clean historical data, holdings, derivatives greeks, liquidity, and intraday risk.

### 28.2 VaR is not a worst-case loss

VaR is a quantile, not a maximum loss.

Losses beyond VaR can be much larger.

### 28.3 Gaussian assumptions

Gaussian VaR can understate tail risk when returns are skewed or fat-tailed.

### 28.4 Cornish-Fisher instability

Cornish-Fisher approximations can behave badly with extreme skew/kurtosis.

### 28.5 Historical VaR assumes the past is representative

Historical VaR can miss new risks or regime shifts.

### 28.6 Monte Carlo depends on the model

Monte Carlo VaR is only as good as the assumed distribution.

### 28.7 One-day horizon

This notebook focuses on one-day risk.

Multi-day risk is not simply one-day risk times $\sqrt{h}$ under fat tails, autocorrelation, and changing volatility.

### 28.8 No liquidity-adjusted VaR

Liquidity costs, bid/ask widening, and liquidation horizons are excluded.

## 29. Research frontier and extensions

Important extensions include:

1. **Filtered historical simulation**  
   Combine EWMA/GARCH volatility with historical residuals.

2. **GARCH VaR**  
   Use conditional volatility forecasts.

3. **Extreme value theory**  
   Generalised Pareto tail modelling.

4. **Expected Shortfall backtesting**  
   More formal ES validation methods.

5. **Multi-day VaR simulation**  
   Simulate full paths rather than square-root-time scaling.

6. **Liquidity-adjusted VaR**  
   Add liquidation cost and market impact.

7. **Delta-gamma VaR**  
   Include nonlinear derivative exposures.

8. **Stress scenario library**  
   Governance-approved historical and hypothetical shocks.

9. **Reverse stress testing**  
   Search for combinations that breach capital limits.

10. **Chinese futures risk**  
   Price limits, night sessions, margin changes, contract rolls, and exchange-specific stress scenarios.

## 30. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_07_risk_parity_and_equal_risk_contribution**  
   Allocate by risk contribution.

2. **04_08_drawdown_control_and_crisis_overlay**  
   Path-dependent risk controls.

3. **04_09_liquidity_and_transaction_cost_risk**  
   Stress costs and liquidation.

4. **04_10_portfolio_constraints_and_margin**  
   Futures margin and leverage limits.

5. **05_02_performance_attribution_report_template**  
   Build risk reports.

6. **07_01_multi_asset_cta_research_pipeline**  
   Integrate VaR/CVaR into a full futures strategy.

## 31. Summary

This notebook implemented VaR, CVaR, and stress testing.

It showed how to:

1. simulate multi-asset portfolio returns with stress regimes;
2. define losses consistently;
3. calculate historical VaR and CVaR;
4. calculate Gaussian parametric VaR and CVaR;
5. calculate Cornish-Fisher modified VaR;
6. compare VaR methods;
7. build rolling VaR forecasts;
8. backtest VaR exceedances;
9. implement a Kupiec-style coverage test;
10. diagnose Expected Shortfall forecasts;
11. calculate EWMA Gaussian VaR;
12. estimate Monte Carlo VaR and CVaR;
13. approximate component VaR;
14. identify historical stress days;
15. build hypothetical stress scenarios;
16. apply PCA stress shocks;
17. run reverse stress tests;
18. build a tail-risk dashboard and governance flags.

The key computational takeaway:

> Tail-risk measurement is a distribution, forecast, and validation problem — not a single number.

The key financial takeaway:

> VaR tells you where the tail starts; CVaR and stress testing tell you why the tail can still hurt.

## 32. Further reading

- Jorion, *Value at Risk.*
- McNeil, Frey, and Embrechts, *Quantitative Risk Management.*
- Acerbi and Tasche on Expected Shortfall.
- Kupiec on VaR backtesting.
- Christoffersen on interval forecast evaluation.
- Basel market risk framework and Expected Shortfall-based capital literature.